---
# Topic 3: File I/O — Reading and Writing Data

File I/O is how your program interacts with the file system — reading datasets, saving trained models, loading configs, writing logs. In AI/ML you constantly read CSV files, load JSON configs, save model weights, and write experiment results.

---

## 3.1 The Context Manager — `with open()`

### 📊 [VISUAL] Context Manager and File Modes

```
WITHOUT context manager (risky):    WITH context manager (correct):

f = open('data.txt', 'r')           with open('data.txt', 'r') as f:
data = f.read()                         data = f.read()
# If error occurs here,             # File is ALWAYS closed after the
# f.close() never runs!             # 'with' block, even if error occurs.
f.close()                           # No need to call f.close().

open() modes:
  'r'   read (default)    — file must exist
  'w'   write             — creates or OVERWRITES
  'a'   append            — creates or adds to end
  'r+'  read and write    — file must exist
  'x'   exclusive create  — fails if file exists
  'b'   binary mode       — add to any: 'rb', 'wb'
```

---

## 3.2 Text Files — Reading and Writing

In [ ]:
# ── WRITING a text file ─────────────────────────────────────────────────────
lines = [
    'Epoch 1: Loss=0.8421, Accuracy=0.6234',
    'Epoch 2: Loss=0.6103, Accuracy=0.7512',
    'Epoch 3: Loss=0.4827, Accuracy=0.8201',
]

with open('training_log.txt', 'w') as f:
    for line in lines:
        f.write(line + '\n')    # write() does NOT add newline automatically

print("Written training_log.txt")

# ── READING — entire file as one string ──────────────────────────────────────
with open('training_log.txt', 'r') as f:
    content = f.read()
print("--- read() ---")
print(content)

# ── READING — list of lines ───────────────────────────────────────────────────
with open('training_log.txt', 'r') as f:
    all_lines = f.readlines()
clean = [line.strip() for line in all_lines]   # remove \n
print("--- readlines() stripped ---")
print(clean)

# ── READING — line by line (most memory-efficient for large files) ────────────
print("--- iterating line by line ---")
with open('training_log.txt', 'r') as f:
    for line in f:
        print(' ', line.strip())

# ── APPENDING ────────────────────────────────────────────────────────────────
with open('training_log.txt', 'a') as f:
    f.write('Epoch 4: Loss=0.3901, Accuracy=0.8745\n')
print("Appended epoch 4.")

## 3.3 CSV Files — The Dataset Format

CSV (Comma-Separated Values) is the most common format for tabular datasets. Python's `csv` module handles it cleanly before you move to Pandas.

In [ ]:
import csv

# ── WRITING a CSV using DictWriter (preferred — named columns) ────────────────
students_dicts = [
    {'name': 'Alice', 'age': 22, 'gpa': 8.5, 'city': 'Mumbai'},
    {'name': 'Bob',   'age': 25, 'gpa': 7.2, 'city': 'Delhi'},
    {'name': 'Carol', 'age': 23, 'gpa': 9.1, 'city': 'Pune'},
]

with open('students.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'age', 'gpa', 'city'])
    writer.writeheader()
    writer.writerows(students_dicts)

print("Written students.csv")

# ── READING a CSV using DictReader ───────────────────────────────────────────
with open('students.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    data = list(reader)     # list of dicts

for row in data:
    print(f"  {row['name']}: GPA {row['gpa']}")

# ── Type conversion — CSV reads EVERYTHING as strings ────────────────────────
with open('students.csv', 'r') as f:
    reader = csv.DictReader(f)
    records = [
        {'name': r['name'], 'age': int(r['age']), 'gpa': float(r['gpa'])}
        for r in reader
    ]

print("\nWith type conversion:")
for r in records:
    print(f"  {r['name']} (age {r['age']}, type={type(r['age']).__name__})")

## 3.4 JSON Files — Configs and API Data

JSON is the universal format for configuration files, API responses, and structured data. In ML, model configs, training parameters, and experiment results are almost always stored as JSON.

> 🔵 **[AI/ML]** JSON is universal in ML engineering:
> - `config.json` — hyperparameters, model architecture settings
> - `results.json` — experiment metrics (MLflow, W&B do this)
> - `annotations.json` — COCO dataset format for object detection labels
> - `model_card.json` — model metadata for deployment (HuggingFace Hub)

In [ ]:
import json

# ── WRITING JSON ─────────────────────────────────────────────────────────────
model_config = {
    'model_name':  'ResNet50',
    'version':     '2.1.0',
    'architecture': {
        'type':    'CNN',
        'layers':  [64, 128, 256, 512],
        'dropout': 0.3
    },
    'training': {
        'lr':         0.001,
        'epochs':     100,
        'batch_size': 32,
        'optimizer':  'Adam'
    },
    'results': {
        'best_accuracy': 0.9432,
        'best_epoch':    47
    }
}

with open('model_config.json', 'w') as f:
    json.dump(model_config, f, indent=4)   # indent=4 -> human-readable

print("Written model_config.json")

# ── READING JSON ─────────────────────────────────────────────────────────────
with open('model_config.json', 'r') as f:
    config = json.load(f)

print(f"Model: {config['model_name']}")
print(f"LR:    {config['training']['lr']}")
print(f"Layers:{config['architecture']['layers']}")

# ── JSON <-> strings ──────────────────────────────────────────────────────────
json_string  = json.dumps(model_config, indent=2)  # dict -> JSON string
back_to_dict = json.loads(json_string)             # JSON string -> dict
print(f"\nRound-trip type: {type(back_to_dict)}")

## 3.5 Working with File Paths (`pathlib`)

In [ ]:
from pathlib import Path

# ── Path construction — cross-platform safe ───────────────────────────────────
base      = Path('data')
train     = base / 'train'           # data/train
val       = base / 'val'             # data/val
model_dir = Path('output') / 'models'

# ── Create directories ────────────────────────────────────────────────────────
train.mkdir(parents=True, exist_ok=True)      # creates data/train/ recursively
model_dir.mkdir(parents=True, exist_ok=True)
print("Directories created.")

# ── File info ─────────────────────────────────────────────────────────────────
f = Path('students.csv')
print(f"name:   {f.name}")      # students.csv
print(f"stem:   {f.stem}")      # students
print(f"suffix: {f.suffix}")    # .csv
print(f"parent: {f.parent}")    # . (current directory)
print(f"exists: {f.exists()}")  # True

# ── Find files by pattern ─────────────────────────────────────────────────────
print("\nJSON files in current dir:")
for json_file in Path('.').glob('*.json'):
    print(f"  {json_file}")

# ── Read/write via pathlib (short files) ──────────────────────────────────────
p = Path('notes.txt')
p.write_text('Hello from pathlib!')
content = p.read_text()
print(f"\npathlib read: {content}")

## ✏️ Exercises — File I/O

**[EXERCISE 5 — Medium]** Write a function `log_experiment(config, results, run_id)` that:
- Creates an `'experiments/'` folder if it doesn't exist
- Saves config as `experiments/run_{id}_config.json`
- Saves results as `experiments/run_{id}_results.json`
- Appends a one-line summary to `experiments/summary.log` (CSV format: `run_id,accuracy,loss`)

**[EXERCISE 6 — Advanced]** Write a `DataLoader` class that: takes a CSV filepath, reads the data in `__init__`, provides `__len__` and `__getitem__` for row access, has `get_column(name)` returning a list, and `summary()` printing column names, row count, and first 3 rows.